Script for builing final run manifest from data-auditing.

Requires master data catalogue file as well as fMRI parameters catalogue.

[Runtime: minimal]

--------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:

import os, re
import datetime
import pandas as pd
from pathlib import Path
import shutil
import numpy as np

In [ ]:
### SET PARAMETERS, AND INPUT & OUTPUT FILEPATHS:

### PARAMS:

MANIFEST_OVERWRITE = config['manifest_overwrite']
HARD_STOP = config['hard_errors']

HAS_QC = not(config['skip_catalogue_QC']) # Should eval to true if QC stage was *not* skipped
GOOD_LABELS = config.get('QC_pass_labels', [])

# Analysis parameters:
CONDITIONS = config['CONDITIONS']
MRI_SELECTION  = config['MRI_selection']
fMRI_SELECTION = config['fMRI_selection']
fMRI_VALID_PRESETS = ['first_any', 'first_good', 'last_any', 'last_good', 'all_good', 'all_any']

COMPLETE_SUBS_ONLY = config['complete_fMRI_data_only']
EXCLUDE_IF_MISSING = config['exclude_sub_if_missing']

EXPORT_SUBSET = config['save_subset_catalogue']

# Type checks (if needed):
if type(MRI_SELECTION) != str:
    raise Exception("ERROR: 'MRI_selection' parameter in config.yaml must be one plain STRING (either a valid preset code or a study time-point).")

# Other params:
TIME_POINTS_DICT = config['session_ID_mappings'] # Needed for chronological session_ID code ordering

# Check to make sure 'EXCLUDE_IF_MISSING' contains only time-points that we are actually selecting for:
assert (
    not EXCLUDE_IF_MISSING
    or all(tp in [s.rstrip('*') for s in fMRI_SELECTION] for tp in EXCLUDE_IF_MISSING)
    or (isinstance(fMRI_SELECTION, str) and "all" in fMRI_SELECTION.lower())
), f"Invalid EXCLUDE_IF_MISSING: {EXCLUDE_IF_MISSING}. Each entry must appear in fMRI_SELECTION (ignoring '*') or fMRI_SELECTION must contain 'all'."

### SET PATHS:
ROOT_DIR = Path(config['root_output_directory'])
DATA_INDEX_PATH = Path(ROOT_DIR) / 'master_data_catalogue.csv'
DATA_INDEX = pd.read_csv(DATA_INDEX_PATH)
PARAMETER_INDEX_PATH = Path(ROOT_DIR) / 'fMRI_parameter_index.csv'
PARAMETER_INDEX = pd.read_csv(PARAMETER_INDEX_PATH)

# # Check to make sure requested fMRI time-points are present in data catalogue:
# if type(fMRI_SELECTION) == str:
#     if fMRI_SELECTION not in fMRI_VALID_PRESETS:
#         string_fMRI_SELECTION = [fMRI_SELECTION.rstrip('*')]
#         [time_point for time_point in string_fMRI_SELECTION if time_point.rstrip('*').lower() not in [column.split('_')[1].lower() for column in DATA_INDEX.columns if 'fMRI_' in column and '_filename' in column]]
# elif type(fMRI_SELECTION) == list:
#     missing_timepoints = [time_point for time_point in fMRI_SELECTION if f"fMRI_{time_point.rstrip('*').lower()}_filename" not in [column.lower() for column in DATA_INDEX.columns]]
# assert not missing_timepoints, f"Invalid fMRI_SELECTION entries (no matching filename columns): {missing_timepoints}"

# Check data catalogue to make sure QC metadata is present, if HAS_QC is set to 'True':
if HAS_QC:
    if (DATA_INDEX['n_good_MRIs'] == '[unknown]').any():
        raise Exception("QC metadata is expected ('skip_catalogue_QC' CONFIG parameter is set to 'False'), but master data catalogue appears to be missing QC metrics."
        f"\n\t--> Check 'master_data_catalogue.csv' in {ROOT_DIR} and ensure it has been successfully decorated with QC metadata.")

print("INPUT \ OUTPUT ===========================\n")
print(f"Main project directory set as: {ROOT_DIR}")
print()
print(f"Master data catalogue set as: {DATA_INDEX_PATH}")
print()
print(f"fMRI parameter catalogue loaded from: {PARAMETER_INDEX_PATH}")
print()
print("SETTINGS ===========================\n")

print(f"OVERWRITE enabled: {MANIFEST_OVERWRITE}\n")

if HARD_STOP:
    print(f"Error mode: 'strict' (hard stops enabled)\n")
else:
    print(f"Error mode: 'soft' (hard stops disabled; errors will drop subjects)\n")

print(f"MRI analysis type: '{MRI_SELECTION}'\n")

if type(fMRI_SELECTION) == list:
    print(f"fMRI analysis type: Time-points\n  --> Study time-points selected for analysis: {fMRI_SELECTION}\n")
elif type(fMRI_SELECTION) == str:
    print(f"fMRI analysis type: '{fMRI_SELECTION}' [PRESET]\n")

print(f"QC metadata tables present?: {HAS_QC}")
if HAS_QC:
    print(f"  --> QC criteria for inclusion in analysis: {GOOD_LABELS}\n")

# Flexibly normalize 'CONDITIONS' parameter to either 'all' (str) or a list of group strings:
if isinstance(CONDITIONS, str):
    if CONDITIONS.strip().lower() == 'all':
        CONDITIONS = 'all'
        print("Analyzing 'all' subject groups (e.g. control + experimental) (as per config.yaml 'CONDITIONS' parameter)")
    else:
        raise Exception("ERROR: 'CONDITIONS' must be 'all' (plain string) or a LIST of group_ID strings.")
elif isinstance(CONDITIONS, list):
    CONDITIONS = [str(x).strip() for x in CONDITIONS]
    if len(CONDITIONS) == 1 and CONDITIONS[0].lower() == 'all':
        CONDITIONS = 'all'
        print("Analyzing 'all' subject groups (e.g. control + experimental) (as per config.yaml 'CONDITIONS' parameter)")
    elif any(x.lower() == 'all' for x in CONDITIONS):
        raise Exception("ERROR: If 'CONDITIONS' is a LIST, it cannot contain 'all' alongside other group names.")
    else:
        print(f"Analysis will include the following subject groups: {CONDITIONS}")
else:
    raise Exception("ERROR: 'CONDITIONS' must be either a string ('all') or a list of group_ID strings.")

------
Subsetting via master_data_catalogue:

Let's do CONDITIONS-based subsetting first, then MRI, then fMRI-based subsetting:

In [ ]:
print(f"Stage 1: Subsetting by experimental condition:")
print(f"\t-Selected conditions: {CONDITIONS}")
if isinstance(CONDITIONS, str) and CONDITIONS.lower() == 'all':
    SUBSET_1 = DATA_INDEX.copy()
else:
    cond_lower = {c.lower() for c in CONDITIONS}
    available = set(DATA_INDEX['group_ID'].dropna().str.lower().unique())
    missing = sorted(cond_lower - available)
    if missing and HARD_STOP:
        raise ValueError(
            f"!!! ERROR: The following condition / subject group(s) were not found in 'group_ID': {missing}\n"
            "Check config.yaml 'CONDITIONS' settings.")
    if missing and not HARD_STOP:
        print(f"!!! WARNING: The following condition / subject group(s) are absent: {missing} — they will be ignored.\n")
    SUBSET_1 = DATA_INDEX[DATA_INDEX['group_ID'].str.lower().isin(cond_lower)]
    if SUBSET_1.empty:
        raise ValueError("After filtering by CONDITIONS, SUBSET is empty.")
SUBSET_1 = SUBSET_1.reset_index(drop=True).copy()
if DATA_INDEX.shape[0] != SUBSET_1.shape[0]:
    print(f"\n  --> Subsetted initial subject pool from {DATA_INDEX.shape[0]} --> {SUBSET_1.shape[0]} subjects:")
else:
    print(f"\n  --> No condition \ experimental group subsetting performed; proceeding from initial subject pool of {DATA_INDEX.shape[0]} subjects.")
display(SUBSET_1)

MRI-based selection:

In [ ]:
# Build MRI_runs from SUBSET_1 according to MRI_SELECTION / HAS_QC / GOOD_LABELS
# Output columns: subject_ID, MRI_session_ID, MRI_filename

MRI_runs_rows = []

# Normalize policy and GOOD_LABELS for robust matching:
_policy = str(MRI_SELECTION).strip().lower().replace("_", " ")
_good = {str(x).strip().lower() for x in GOOD_LABELS}

# QC-dependent policies -- must have HAS_QC == True:
if not HAS_QC and _policy in ("first good", "last good"):
    raise ValueError(
        f"Requested MRI_SELECTION='{MRI_SELECTION}' requires QC metrics, "
        "but HAS_QC is False (QC columns are not usable). "
        "Choose a non-QC policy (e.g., 'first_any' or 'last_any') or provide valid QC.")

# Use only timepoints that actually have MRI filename columns:
ordered_tps = [tp for tp in TIME_POINTS_DICT.keys()
               if f"MRI_{tp}_filename" in SUBSET_1.columns]

for _, row in SUBSET_1.iterrows():

    if _policy == "first any":
        for tp in ordered_tps:
            fn = row.get(f"MRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                MRI_runs_rows.append({"subject_ID": row["subject_ID"], "MRI_session_ID": tp, "MRI_filename": fn})
                break

    elif _policy == "last any":
        for tp in reversed(ordered_tps):
            fn = row.get(f"MRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                MRI_runs_rows.append({"subject_ID": row["subject_ID"], "MRI_session_ID": tp, "MRI_filename": fn})
                break

    elif _policy == "first good":
        for tp in ordered_tps:
            fn = row.get(f"MRI_{tp}_filename", pd.NA)
            qc = row.get(f"MRI_{tp}_QC_summary", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            qc = str(qc).strip().lower() if pd.notna(qc) else ""
            if fn and (qc in _good):
                MRI_runs_rows.append({"subject_ID": row["subject_ID"], "MRI_session_ID": tp, "MRI_filename": fn})
                break

    elif _policy == "last good":
        for tp in reversed(ordered_tps):
            fn = row.get(f"MRI_{tp}_filename", pd.NA)
            qc = row.get(f"MRI_{tp}_QC_summary", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            qc = str(qc).strip().lower() if pd.notna(qc) else ""
            if fn and (qc in _good):
                MRI_runs_rows.append({"subject_ID": row["subject_ID"], "MRI_session_ID": tp, "MRI_filename": fn})
                break

    else:
        # Treat MRI_SELECTION as a specific timepoint string (case-insensitive):
        raw_tp = str(MRI_SELECTION).strip()
        bypass_qc = raw_tp.endswith("*")
        # Remove '*' if present:
        tp_search = raw_tp.rstrip("*")
        # Canonicalize for matching (ignore _ and case):
        tp_norm = tp_search.lower().replace("_", " ")

        # Find matching timepoint key:
        tp_match = None
        for k in ordered_tps:
            if k.lower().replace("_", " ") == tp_norm:
                tp_match = k
                break

        if tp_match is not None:
            fn = row.get(f"MRI_{tp_match}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""

            if fn:  # <-- non-empty filename required

                # Case 1: bypass QC requested via '*' suffix:
                if bypass_qc:
                    MRI_runs_rows.append({
                        "subject_ID": row["subject_ID"],
                        "MRI_session_ID": tp_match,
                        "MRI_filename": fn})

                # Case 2: QC required and QC available:
                elif HAS_QC and f"MRI_{tp_match}_QC_summary" in SUBSET_1.columns:
                    qc = row.get(f"MRI_{tp_match}_QC_summary", pd.NA)
                    qc = str(qc).strip().lower() if pd.notna(qc) else ""
                    if qc in _good:
                        MRI_runs_rows.append({
                            "subject_ID": row["subject_ID"],
                            "MRI_session_ID": tp_match,
                            "MRI_filename": fn})

                # Case 3: HAS_QC == False → allow any non-empty filename:
                elif not HAS_QC:
                    MRI_runs_rows.append({
                        "subject_ID": row["subject_ID"],
                        "MRI_session_ID": tp_match,
                        "MRI_filename": fn})



MRI_runs = pd.DataFrame(MRI_runs_rows)

# Optional quick sanity print if output winds up empty:
if MRI_runs.empty:
    print("[info] MRI_runs is empty.")
    print(" - Policy (normalized):", _policy)
    print(" - Ordered timepoints (with filename cols):", ordered_tps[:10], "..." if len(ordered_tps)>10 else "")

display(SUBSET_1.shape[0])
display(MRI_runs.shape[0])
MRI_runs

"Re-harmonize" updated subject_IDs to SUBSET_2:

In [ ]:
# Re-harmonize to include only subjects that survived MRI selection:
SUBSET_2 = SUBSET_1[SUBSET_1["subject_ID"].isin(MRI_runs["subject_ID"])].copy()

print(f"SUBSET_1: {SUBSET_1.shape[0]} subjects → SUBSET_2: {SUBSET_2.shape[0]} subjects after MRI filtering")

Next, we perform fMRI selection:

In [ ]:
# =========================
# Build fMRI_runs from SUBSET_2 according to fMRI_SELECTION / HAS_QC / GOOD_LABELS
# - Presets: first_good, last_good, first_any, last_any, all_good, all_any
# - List of timepoints allowed; '*' suffix on a timepoint bypasses QC for that timepoint
# - Always include all subjects (one row per subject)
# - Output:
#     * single-file presets -> long format:   [subject_ID, session_ID, fMRI_filename]
#     * multi-file presets or list -> wide format: [subject_ID, fMRI_<tp>_filename ...]
# =========================

# Normalize preset/control inputs:
_preset = None
if isinstance(fMRI_SELECTION, str):
    _preset = fMRI_SELECTION.strip().lower().replace("_", " ")
_good = {str(x).strip().lower() for x in GOOD_LABELS}

# Available fMRI timepoints (preserve chronological order):
ordered_tps = [tp for tp in TIME_POINTS_DICT.keys()
               if f"fMRI_{tp}_filename" in SUBSET_2.columns]

def _canon(s):  # <-- inline canonicalizer function
    return str(s).strip().lower().replace("_", " ")

# Determine which timepoints we will actually look for ('selected_tps'):
selected_tps = []        # <-- list of canonical df keys (actual labels from 'ordered_tps')
tp_bypass_qc = dict()    # <-- {tp -> True/False} (bypass when '*' used per-tp, list mode only)

if isinstance(fMRI_SELECTION, list):
    # Map requested list to actual keys; honor QC-bypassing '*' modifier per entry:
    for raw in fMRI_SELECTION:
        s = str(raw).strip()
        bypass = s.endswith("*")
        s_clean = s.rstrip("*")
        norm = _canon(s_clean)
        match = None
        for k in ordered_tps:
            if _canon(k) == norm:
                match = k
                break
        if match is not None:
            if match not in selected_tps:
                selected_tps.append(match)
            tp_bypass_qc[match] = tp_bypass_qc.get(match, False) or bypass
    if not selected_tps:
        raise ValueError(
            f"No recognized fMRI timepoints in list {fMRI_SELECTION}. "
            f"Available: {ordered_tps}")

elif isinstance(fMRI_SELECTION, str):
    if _preset in ("all any", "all good"):
        selected_tps = ordered_tps[:]  # <-- all available time-points
    elif _preset in ("first any", "last any", "first good", "last good"):
        selected_tps = []  # <-- NB: single-file policies don't pre-declare time-points
    else:
        raise ValueError(
            f"Invalid fMRI_SELECTION preset: '{fMRI_SELECTION}'. Use one of "
            "'first_good', 'last_good', 'first_any', 'last_any', 'all_good', 'all_any', "
            "or pass a LIST of time-points (each may end with '*' to bypass QC).")

# CHECK: QC-dependent presets must have QC data available:
if isinstance(fMRI_SELECTION, str) and _preset in ("first good", "last good", "all good") and not HAS_QC:
    raise ValueError(
        f"Requested fMRI_SELECTION='{fMRI_SELECTION}' requires QC metrics, "
        "but HAS_QC is False. Choose 'first_any'/'last_any'/'all_any' or provide QC.")

# Decide output shape (long vs wide):
subject_ids = SUBSET_2["subject_ID"].tolist()

if isinstance(fMRI_SELECTION, list):
    multi_output = len(selected_tps) > 1
else:
    multi_output = (_preset in ("all any", "all good"))

if not multi_output:
    # Long format: one file per subject (or else NaN):
    fMRI_runs = pd.DataFrame({"subject_ID": subject_ids,
                              "fMRI_session_ID": np.nan,
                              "fMRI_filename": np.nan})
else:
    # Wide format: only columns for selected time-points (not every possible time-point):
    cols = ["subject_ID"] + [f"fMRI_{tp}_filename" for tp in selected_tps]
    fMRI_runs = pd.DataFrame(index=range(len(subject_ids)), columns=cols)
    fMRI_runs["subject_ID"] = subject_ids

# --- Populate according to selection policy ---
if isinstance(fMRI_SELECTION, list):
    # Strict list mode: QC required unless '*' used (bypass), and HAS_QC must be True to enforce:
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        if not multi_output:
            # Exactly one tp requested → behave like single-file selection (long format):
            tp = selected_tps[0]
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                if tp_bypass_qc.get(tp, False):
                    fMRI_runs.at[i, "fMRI_session_ID"] = tp
                    fMRI_runs.at[i, "fMRI_filename"] = fn
                else:
                    if HAS_QC and f"fMRI_{tp}_QC_summary" in SUBSET_2.columns:
                        qc = row.get(f"fMRI_{tp}_QC_summary", pd.NA)
                        qc = str(qc).strip().lower() if pd.notna(qc) else ""
                        if qc in _good:
                            fMRI_runs.at[i, "fMRI_session_ID"] = tp
                            fMRI_runs.at[i, "fMRI_filename"] = fn
                    elif not HAS_QC:
                        # No QC available → treat list entries as if they had '*' (accept any non-empty filename)
                        fMRI_runs.at[i, "fMRI_session_ID"] = tp
                        fMRI_runs.at[i, "fMRI_filename"] = fn
                    else:
                        pass  # QC is expected but missing column → keep strict behavior

        else:
            # Multiple tps selected → wide format:
            for tp in selected_tps:
                fn = row.get(f"fMRI_{tp}_filename", pd.NA)
                fn = str(fn).strip() if pd.notna(fn) else ""
                if not fn:
                    continue
                if tp_bypass_qc.get(tp, False):
                    fMRI_runs.at[i, f"fMRI_{tp}_filename"] = fn
                else:
                    if HAS_QC and f"fMRI_{tp}_QC_summary" in SUBSET_2.columns:
                        qc = row.get(f"fMRI_{tp}_QC_summary", pd.NA)
                        qc = str(qc).strip().lower() if pd.notna(qc) else ""
                        if qc in _good:
                            fMRI_runs.at[i, f"fMRI_{tp}_filename"] = fn
                    elif not HAS_QC:
                        # No QC available → treat list entries as if they had '*' (accept any non-empty filename)
                        fMRI_runs.at[i, f"fMRI_{tp}_filename"] = fn
                    else:
                        pass  # QC is expected but missing column → keep strict behavior


elif _preset == "all any":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in selected_tps:  # all available tps
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                fMRI_runs.at[i, f"fMRI_{tp}_filename"] = fn

elif _preset == "all good":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in selected_tps:  # all available tps
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            qc = row.get(f"fMRI_{tp}_QC_summary", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            qc = str(qc).strip().lower() if pd.notna(qc) else ""
            if fn and (qc in _good):
                fMRI_runs.at[i, f"fMRI_{tp}_filename"] = fn

elif _preset == "first any":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in ordered_tps:
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                fMRI_runs.at[i, "fMRI_session_ID"] = tp
                fMRI_runs.at[i, "fMRI_filename"] = fn
                break

elif _preset == "last any":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in reversed(ordered_tps):
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            if fn:
                fMRI_runs.at[i, "fMRI_session_ID"] = tp
                fMRI_runs.at[i, "fMRI_filename"] = fn
                break

elif _preset == "first good":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in ordered_tps:
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            qc = row.get(f"fMRI_{tp}_QC_summary", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            qc = str(qc).strip().lower() if pd.notna(qc) else ""
            if fn and (qc in _good):
                fMRI_runs.at[i, "fMRI_session_ID"] = tp
                fMRI_runs.at[i, "fMRI_filename"] = fn
                break

elif _preset == "last good":
    for i, (_, row) in enumerate(SUBSET_2.iterrows()):
        for tp in reversed(ordered_tps):
            fn = row.get(f"fMRI_{tp}_filename", pd.NA)
            qc = row.get(f"fMRI_{tp}_QC_summary", pd.NA)
            fn = str(fn).strip() if pd.notna(fn) else ""
            qc = str(qc).strip().lower() if pd.notna(qc) else ""
            if fn and (qc in _good):
                fMRI_runs.at[i, "fMRI_session_ID"] = tp
                fMRI_runs.at[i, "fMRI_filename"] = fn
                break

# Final tidying for wide mode (drop any all-empty columns, just in case):
if multi_output:
    to_drop = [c for c in fMRI_runs.columns if c != "subject_ID" and fMRI_runs[c].isna().all()]
    if to_drop:
        fMRI_runs = fMRI_runs.drop(columns=to_drop)

# Print concise summary:
if multi_output:
    print(f"[fMRI_runs] wide format: {fMRI_runs.shape[0]} subjects, {fMRI_runs.shape[1]-1} timepoint columns.")
else:
    hits = fMRI_runs["fMRI_filename"].notna().sum()
    print(f"[fMRI_runs] long format: {fMRI_runs.shape[0]} subjects, {hits} with a selected file.")

In [ ]:
# Rows w/ NaNs here indicate subjects lacking data meeting the defined criteria (& will be dropped from final manifest): 
fMRI_runs

Filter down 'fMRI_runs' based on missing data tolerances & other analysis settings:

In [ ]:
# =========================
# Post-selection drops on fMRI_runs + harmonize to SUBSET_3
# Rules:
# 1) Drop rows with no fMRI files at all (all filename cols empty/NaN)  [always]
# 2) If COMPLETE_SUBS_ONLY == True → keep only rows where ALL filename cols are present
# 3) If EXCLUDE_IF_MISSING is a non-empty list of timepoints → drop rows missing any of those specific tps
# Finally: SUBSET_3 = SUBSET_2 restricted to surviving subject_IDs
# =========================

# Identify filename columns (works for wide OR long table format):
filename_cols = [c for c in fMRI_runs.columns if c.startswith("fMRI_") and c.endswith("_filename")]
if not filename_cols and "fMRI_filename" in fMRI_runs.columns:
    filename_cols = ["fMRI_filename"]  # long format

if not filename_cols:
    raise ValueError("No fMRI filename columns found in fMRI_runs.")

# Helper: boolean DataFrame indicating non-empty filenames (case-insensitive, trims stray whitespaces):
_nonempty = fMRI_runs[filename_cols].applymap(
    lambda v: (isinstance(v, str) and v.strip() != "") or (pd.notna(v) and str(v).strip() != ""))

# Hard drop: rows with zero files across all selected columns:
drop_mask_allempty = ~_nonempty.any(axis=1)
n_drop_allempty = int(drop_mask_allempty.sum())
if n_drop_allempty:
    print(f"[drop] Subjects with no fMRI files at all (given selection): {n_drop_allempty}")
    fMRI_runs = fMRI_runs.loc[~drop_mask_allempty].reset_index(drop=True)
    _nonempty = _nonempty.loc[~drop_mask_allempty].reset_index(drop=True)
else:
    print("[drop] No rows dropped for 'no fMRI files at all'")

# COMPLETE_SUBS_ONLY: keep only rows with all filename cols present:
if COMPLETE_SUBS_ONLY:
    drop_mask_incomplete = ~_nonempty.all(axis=1)
    n_drop_incomplete = int(drop_mask_incomplete.sum())
    if n_drop_incomplete:
        print(f"[drop] COMPLETE_SUBS_ONLY == 'True' → subjects missing any selected time-point: {n_drop_incomplete}")
        fMRI_runs = fMRI_runs.loc[~drop_mask_incomplete].reset_index(drop=True)
        _nonempty = _nonempty.loc[~drop_mask_incomplete].reset_index(drop=True)
    else:
        print("[drop] COMPLETE_SUBS_ONLY == 'True' → no subjects were incomplete")
else:
    print("[drop] COMPLETE_SUBS_ONLY == 'False' → no action")

# EXCLUDE_IF_MISSING: list of mandatory timepoints to enforce:
# (Accepts "falsy" values (i.e. None/False/[]/np.nan) == no action)
if EXCLUDE_IF_MISSING:
    # Normalize to list of canonical timepoint keys present among 'filename_cols':
    def _canon(s): return str(s).strip().lower().replace("_", " ")
    # Map filename columns → timepoint labels (wide format) or to the single 'fMRI_session_ID' (long format):
    if filename_cols == ["fMRI_filename"]:
        # Long format: we only have one filename column; enforce only if the requested
        # mandatory timepoint equals the row's fMRI_session_ID (so we must check per-row).
        # Build a per-row mask: drop if row's fMRI_session_ID matches any mandatory tp and the filename is empty.
        # First, figure canonical mandatory set:
        mandatory_set = {_canon(tp) for tp in EXCLUDE_IF_MISSING}
        # canonicalize row fMRI_session_ID:
        ses_col = "fMRI_session_ID" if "fMRI_session_ID" in fMRI_runs.columns else None
        if ses_col is None:
            raise ValueError("Long-format fMRI_runs is missing 'fMRI_session_ID' needed for EXCLUDE_IF_MISSING checks.")
        # Build row-wise mask: mandatory match AND filename empty:
        ses_canon = fMRI_runs[ses_col].astype(str).str.lower().str.replace("_", " ").str.strip()
        filename_empty = ~_nonempty["fMRI_filename"]
        drop_mask_mand = ses_canon.isin(mandatory_set) & filename_empty
        n_drop_mand = int(drop_mask_mand.sum())
        if n_drop_mand:
            print(f"[drop] EXCLUDE_IF_MISSING={list(EXCLUDE_IF_MISSING)} → dropped (long-format, matching fMRI_session_ID): {n_drop_mand}")
            fMRI_runs = fMRI_runs.loc[~drop_mask_mand].reset_index(drop=True)
            _nonempty = _nonempty.loc[~drop_mask_mand].reset_index(drop=True)
        else:
            print(f"[drop] EXCLUDE_IF_MISSING={list(EXCLUDE_IF_MISSING)} → no rows dropped (long-format)")
    else:
        # Wide format: enforce that each mandatory tp column (if present) is non-empty:
        # Map requested tps -> actual existing filename columns:
        requested = list(EXCLUDE_IF_MISSING)
        # Build mapping from canonical tp to actual column name:
        tp_to_col = {}
        for col in filename_cols:
            # col format: fMRI_<tp>_filename:
            tp = col[len("fMRI_") : -len("_filename")]
            tp_to_col[_canon(tp)] = col

        required_cols = [tp_to_col[_canon(tp)] for tp in requested if _canon(tp) in tp_to_col]

        # Warn (soft) if some requested mandatory tps are not in the selected columns:
        missing_required = [tp for tp in requested if _canon(tp) not in tp_to_col]
        if missing_required:
            print(f"[warn] EXCLUDE_IF_MISSING includes timepoints not present in selection: {missing_required} (ignored)")

        if required_cols:
            # Drop rows where any required col is empty:
            req_nonempty = _nonempty[required_cols]
            drop_mask_mand = ~req_nonempty.all(axis=1)
            n_drop_mand = int(drop_mask_mand.sum())
            if n_drop_mand:
                print(f"[drop] EXCLUDE_IF_MISSING enforced on {len(required_cols)} tp(s) → dropped: {n_drop_mand}")
                fMRI_runs = fMRI_runs.loc[~drop_mask_mand].reset_index(drop=True)
                _nonempty = _nonempty.loc[~drop_mask_mand].reset_index(drop=True)
            else:
                print(f"[drop] EXCLUDE_IF_MISSING enforced on {len(required_cols)} tp(s) → no rows dropped")
        else:
            print("[drop] EXCLUDE_IF_MISSING → no matching selected columns; no action")
else:
    print("[drop] EXCLUDE_IF_MISSING not set → no action")

In [ ]:
# Harmonize back to SUBSET_3:
SUBSET_3 = SUBSET_2[SUBSET_2["subject_ID"].isin(fMRI_runs["subject_ID"])].copy()
print(f"[harmonize] SUBSET_2 → SUBSET_3: {SUBSET_2.shape[0]} → {SUBSET_3.shape[0]} subjects")
SUBSET_3.sample(10)

Next we "un-pack" the 'fMRI_runs' dataframe so that we have one row for every file, onto which we will join the corresponding file-level fMRI parameters from 'PARAMETER_INDEX':

In [ ]:
# =========================
# Un-pack fMRI_runs → one row per file, then join PARAMETER_INDEX
# + Sanity check: count of non-empty filenames in fMRI_runs == rows in fMRI_parameters
# + Audit: check PARAMETER_INDEX duplicates on merge keys, verify session_ID consistency, drop session_ID_param
# + Validate required parameters; HARD_STOP or drop offending files from fMRI_runs
# Output:
#   fMRI_parameters: subject_ID, session_ID, fMRI_filename, (… + all PARAMETER_INDEX cols via left-merge)
# =========================

# Detect format and build long rows [subject_ID, session_ID, fMRI_filename]:
if "fMRI_filename" in fMRI_runs.columns:
    # Long format already: rename fMRI_session_ID -> session_ID for consistency
    required_cols = ["subject_ID", "fMRI_session_ID", "fMRI_filename"]
    missing = [c for c in required_cols if c not in fMRI_runs.columns]
    if missing:
        raise ValueError(f"Long-format fMRI_runs missing required columns: {missing}")

    fMRI_parameters = (
        fMRI_runs[["subject_ID", "fMRI_session_ID", "fMRI_filename"]]
        .rename(columns={"fMRI_session_ID": "session_ID"}).copy())
else:
    # Wide format → melt only filename columns we actually have:
    fn_cols = [c for c in fMRI_runs.columns if c.startswith("fMRI_") and c.endswith("_filename")]
    if not fn_cols:
        raise ValueError("No fMRI filename columns found in fMRI_runs.")
    long = (
        fMRI_runs.melt(
            id_vars=["subject_ID"],
            value_vars=fn_cols,
            var_name="wide_col",
            value_name="fMRI_filename"))
    # Extract session_ID from column name: fMRI_<time-point>_filename:
    long["session_ID"] = long["wide_col"].str[len("fMRI_") : -len("_filename")]
    fMRI_parameters = long[["subject_ID", "session_ID", "fMRI_filename"]].copy()

# Cleanup: treat empty strings as NaN, trim whitespaces:
fMRI_parameters["fMRI_filename"] = (
    fMRI_parameters["fMRI_filename"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "NA": np.nan, "nan": np.nan}))

# Drop rows with no filename (some subjects may have none after prior filters):
before = fMRI_parameters.shape[0]
fMRI_parameters = fMRI_parameters.dropna(subset=["fMRI_filename"]).reset_index(drop=True)
after = fMRI_parameters.shape[0]
dropped_empty = before - after
if dropped_empty:
    print(f"[info] Dropped {dropped_empty} rows with empty/NaN fMRI_filename during un-pack.")

# === Sanity check: unpack consistency ===
# Count total non-empty filename entries in fMRI_runs (across all *_filename cols):
filename_cols = [c for c in fMRI_runs.columns if c.startswith("fMRI_") and c.endswith("_filename")]
if not filename_cols and "fMRI_filename" in fMRI_runs.columns:
    filename_cols = ["fMRI_filename"]

def _is_nonempty(v):
    if pd.isna(v):
        return False
    s = str(v).strip()
    return s not in {"", "NA", "nan"}

nonempty_total = sum(fMRI_runs[c].apply(_is_nonempty).sum() for c in filename_cols)
row_count = len(fMRI_parameters)

assert nonempty_total == row_count, (
    f"Sanity check failed: total non-empty filenames in fMRI_runs ({nonempty_total}) "
    f"≠ number of rows in fMRI_parameters ({row_count}).")
print(f"[sanity-check] Non-empty fMRI filenames in fMRI_runs = {nonempty_total}  |  "
      f"Rows in fMRI_parameters = {row_count}  [OK]")

# Left-join PARAMETER_INDEX to restrict to selected files (by subject_ID + fMRI_filename):
PARAM_JOIN = PARAMETER_INDEX.copy()
PARAM_JOIN["fMRI_filename"] = PARAM_JOIN["fMRI_filename"].astype(str).str.strip()

# Check duplicates on merge keys to avoid one-to-many merges:
_dup_count = PARAM_JOIN.duplicated(subset=["subject_ID", "fMRI_filename"]).sum()
if _dup_count:
    print(f"[warn] PARAMETER_INDEX has {_dup_count} duplicate key rows for ['subject_ID','fMRI_filename'] "
          f"→ merge may create duplicates. Consider de-duplicating upstream.")

fMRI_parameters = fMRI_parameters.merge(
    PARAM_JOIN,
    how="left",
    on=["subject_ID", "fMRI_filename"],
    suffixes=("", "_param"))
fMRI_parameters = fMRI_parameters.sort_values(by=['subject_ID', 'session_ID']).reset_index(drop=True).copy()

# Audit session_ID consistency (left vs PARAMETER_INDEX), then drop 'session_ID_param':
if "session_ID_param" in fMRI_parameters.columns:
    left_norm  = fMRI_parameters["session_ID"].astype(str).str.strip().str.lower().str.replace("_", " ", regex=False)
    right_norm = fMRI_parameters["session_ID_param"].astype(str).str.strip().str.lower().str.replace("_", " ", regex=False)
    mismatch_mask = left_norm.notna() & right_norm.notna() & (left_norm != right_norm)
    n_mismatch = int(mismatch_mask.sum())
    if n_mismatch:
        print(f"[warn] session_ID mismatch between fMRI_runs (left) and PARAMETER_INDEX (right): {n_mismatch} row(s). Showing up to 10:")
        display(
            fMRI_parameters.loc[mismatch_mask, ["subject_ID", "fMRI_filename", "session_ID", "session_ID_param"]]
            .head(10))
    # Keep the unpacked session_ID as canonical; drop the PARAM copy:
    fMRI_parameters = fMRI_parameters.drop(columns=["session_ID_param"])

# Validate required parameters and either HARD_STOP or drop offending files from fMRI_runs:
# Define list of columns allowed to be empty:
allowed_empty = {
    'TotalReadoutTime','EffectiveEchoSpacing','SliceTiming_len','SliceTiming_unique_count',
    'Manufacturer','ManufacturersModelName','site_code','protocol_code'}
# Meta/join columns we don't validate for emptiness:
meta_cols = {'subject_ID','group_ID','session_ID','fMRI_filename'}

# Determine required columns present in the merged frame:
present_param_cols = [c for c in PARAMETER_INDEX.columns if c in fMRI_parameters.columns]
required_cols = [c for c in present_param_cols if c not in allowed_empty and c not in meta_cols]

# Build completeness mask over required columns:
def _req_ok(v):
    if pd.isna(v):
        return False
    s = str(v).strip()
    return s != "" and s.lower() not in {"na","nan","none"}

incomplete_mask = ~fMRI_parameters[required_cols].applymap(_req_ok).all(axis=1) if required_cols else pd.Series(False, index=fMRI_parameters.index)
n_incomplete = int(incomplete_mask.sum())

print(f"[validate] Required-parameter check: {len(required_cols)} fields enforced; "
      f"{n_incomplete} file(s) incomplete.")

if n_incomplete:
    # Summarize missing fields (top offenders, if many):
    missing_per_col = (~fMRI_parameters[required_cols].applymap(_req_ok)).sum().sort_values(ascending=False)
    top_missing = missing_per_col[missing_per_col > 0].head(10)
    if len(top_missing):
        print("[validate] Top missing fields (count of files missing):")
        for col, cnt in top_missing.items():
            print(f"  - {col}: {int(cnt)}")

    if HARD_STOP:
        raise Exception(
            f"ERROR: {n_incomplete} fMRI file(s) missing required parameters. "
            "Provide/repair PARAMETER_INDEX or relax settings.")
    else:
        print(f"[WARNING] Dropping {n_incomplete} offending fMRI file(s) from fMRI_runs (by filename).")

        # Offending filenames set (trimmed strings):
        offending_filenames = set(
            fMRI_parameters.loc[incomplete_mask, "fMRI_filename"].astype(str).str.strip())

        # Remove by simple filename match:
        if "fMRI_filename" in fMRI_runs.columns:
            # Long format → drop rows where fMRI_filename is in offending set:
            before_rows = len(fMRI_runs)
            fMRI_runs = fMRI_runs[~fMRI_runs["fMRI_filename"].astype(str).str.strip().isin(offending_filenames)].reset_index(drop=True)
            after_rows = len(fMRI_runs)
            print(f"[drop] fMRI_runs (long): removed {before_rows - after_rows} row(s).")
        else:
            # Wide format → null out any cells equal to offending filenames:
            fn_cols = [c for c in fMRI_runs.columns if c.startswith("fMRI_") and c.endswith("_filename")]
            before_cells = sum(fMRI_runs[c].astype(str).str.strip().isin(offending_filenames).sum() for c in fn_cols)
            for c in fn_cols:
                mask = fMRI_runs[c].astype(str).str.strip().isin(offending_filenames)
                fMRI_runs.loc[mask, c] = pd.NA
            after_cells = sum(fMRI_runs[c].astype(str).str.strip().isin(offending_filenames).sum() for c in fn_cols)
            print(f"[drop] fMRI_runs (wide): cleared {before_cells - after_cells} filename cell(s).")

# Reorder columns:
preferred_order = ["subject_ID", "group_ID", "session_ID", "site_code", "protocol_code", "fMRI_filename", "fMRI_basepath"]
fMRI_parameters = fMRI_parameters[[c for c in preferred_order if c in fMRI_parameters.columns] +
                                  [c for c in fMRI_parameters.columns if c not in preferred_order]]

print(f"\nCompiled parameters for {fMRI_parameters.shape[0]} fMRI files (across {fMRI_parameters.subject_ID.nunique()} subject_IDs):")
fMRI_parameters

Should have everything we need now; let's compile the final dataframes and save / export:

In [ ]:
# =========================
# Merge MRI_runs + fMRI_runs → run_manifest (anchor on fMRI subjects)
# Adds group_ID (from PARAMETER_INDEX) as second column.
# =========================

# Collect filename columns:
mri_cols  = [c for c in MRI_runs.columns  if c.startswith("MRI_")]
fmri_cols = [c for c in fMRI_runs.columns if c.startswith("fMRI_")]

if "MRI_filename" in MRI_runs.columns and "MRI_filename" not in mri_cols:
    mri_cols.append("MRI_filename")
if "fMRI_filename" in fMRI_runs.columns and "fMRI_filename" not in fmri_cols:
    fmri_cols.append("fMRI_filename")

# Reduce to subject_ID + filename cols:
mri_df  = MRI_runs[["subject_ID"] + mri_cols].copy()  if mri_cols  else MRI_runs[["subject_ID"]].copy()
fmri_df = fMRI_runs[["subject_ID"] + fmri_cols].copy() if fmri_cols else fMRI_runs[["subject_ID"]].copy()

# Merge anchored on fMRI subjects:
run_manifest = fmri_df.merge(mri_df, on="subject_ID", how="left", suffixes=("", "_MRIdup"))

# Add 'group_ID' column data from PARAMETER_INDEX:
if "group_ID" in PARAMETER_INDEX.columns:
    group_map = PARAMETER_INDEX[["subject_ID", "group_ID"]].drop_duplicates()
    run_manifest = run_manifest.merge(group_map, on="subject_ID", how="left")

# Reorder columns: subject_ID, group_ID, MRI..., fMRI...:
ordered_cols = ["subject_ID"]
if "group_ID" in run_manifest.columns:
    ordered_cols.append("group_ID")
ordered_cols += [c for c in run_manifest.columns if c in mri_cols]
ordered_cols += [c for c in run_manifest.columns if c in fmri_cols]
run_manifest = run_manifest.reindex(columns=ordered_cols)

print(f"[run_manifest] {run_manifest.shape[0]} subjects; {len(mri_cols)} MRI col(s), {len(fmri_cols)} fMRI col(s).")
display(run_manifest.group_ID.value_counts())
display(run_manifest.head(10))

--------
#### Final save / export:

In [ ]:
# =========================
# Export manifests (w/ conditional overwrite)
# =========================

if EXPORT_SUBSET:
    subset_export_filepath = Path(ROOT_DIR) / "run_data_catalog.csv"
    if not subset_export_filepath.exists() or MANIFEST_OVERWRITE:
        SUBSET_3.to_csv(subset_export_filepath, index=False)
        print(f"[export] Saved subset catalogue → {subset_export_filepath.name}")
    else:
        print(f"[skip] {subset_export_filepath.name} already exists; skipping (set MANIFEST_OVERWRITE=True to overwrite).")

run_manifest_filepath  = Path(ROOT_DIR) / "subject_manifest.csv"
fMRI_manifest_filepath = Path(ROOT_DIR) / "fMRI_manifest.csv"
for df, path in [(run_manifest, run_manifest_filepath), (fMRI_parameters, fMRI_manifest_filepath)]:
    if not path.exists() or MANIFEST_OVERWRITE:
        df.to_csv(path, index=False)
        print(f"[export] Saved {path.name}")
    else:
        print(f"[skip] {path.name} already exists; skipping (set MANIFEST_OVERWRITE=True to overwrite).")